# SPAN-1 — Step 1: Load & Verify the Portfolio

**Goal:** Read `Portfolio_Agri.xlsx`, clean it up, and compute two things per position:
- `units_per_lot` → how many quote units fit in one lot (needed to get contract value in INR)
- `contract_value_INR` → the full INR value of **one lot** at the current settle price

We also assign a **signed lot** number (+ve for Long, −ve for Short).

---
### Why does `units_per_lot` matter?
MCX quotes prices in **different units** across commodities:

| Commodity | Price Quoted | Lot Size | Units per Lot |
|-----------|-------------|----------|---------------|
| Cotton Seed Oilcake | per 20 kg bag | 10 MT | 10 MT ÷ 0.02 MT = **500 bags** |
| All others | per Quintal | 3–10 MT | lot_size_MT × 10 quintals/MT |

Contract Value = Settle Price × Units per Lot

In [ ]:
# Install if running fresh on Colab
!pip install openpyxl --quiet

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('✓ Libraries ready')

In [ ]:
# ── STEP 1A: Read the Excel file ─────────────────────────────────────────────

EXCEL_FILE = 'Portfolio_Agri.xlsx'   # make sure this is uploaded to Colab

raw = pd.read_excel(EXCEL_FILE)

# Clean column names (Excel has newlines inside headers)
raw.columns = raw.columns.str.replace('\n', ' ', regex=False).str.strip()

print('Raw columns detected:')
for col in raw.columns:
    print(f'  → "{col}"')

print('\nRaw data (including summary row):')
display(raw)

In [ ]:
# ── STEP 1B: Clean & rename ───────────────────────────────────────────────────

# Drop the 'PORTFOLIO TOTAL' summary row — it's not a real position
df = raw[raw['Commodity'].notna() & (raw['Commodity'] != 'PORTFOLIO TOTAL')].copy()
df = df.reset_index(drop=True)

# Rename columns to short, code-friendly names
df = df.rename(columns={
    'Lot Size (MT)':       'lot_size_MT',
    'Quote Unit':          'quote_unit',
    'Expiry Cycle':        'expiry_cycle',
    'Mar-26 Settle (INR)': 'settle_price',
    'Lots':                'lots',
    'Direction':           'direction',
    'Notional (INR)':      'notional_INR'
})

print(f'✓ {len(df)} active positions loaded (PORTFOLIO TOTAL row dropped)')
display(df)

In [ ]:
# ── STEP 1C: Compute units_per_lot ───────────────────────────────────────────
#
# Rule:
#   'per Quintal' → 1 MT = 10 quintals  → units_per_lot = lot_size_MT × 10
#   'per 20 kg'   → 1 MT = 50 bags      → units_per_lot = lot_size_MT × 50
#
# Example check:
#   Cotton Seed Oilcake: lot=10 MT, per 20 kg → 10 × 50 = 500 bags
#   Guar Seed:           lot=10 MT, per quintal → 10 × 10 = 100 quintals

def get_units_per_lot(row):
    qu = str(row['quote_unit']).strip().lower()
    ls = row['lot_size_MT']
    if 'quintal' in qu:
        return ls * 10
    elif '20 kg' in qu:
        return ls * 50
    else:
        return ls   # fallback: price per MT

df['units_per_lot'] = df.apply(get_units_per_lot, axis=1)

print('Units per lot (how many quote-units fit in one lot):')
display(df[['Commodity', 'lot_size_MT', 'quote_unit', 'units_per_lot']])

In [ ]:
# ── STEP 1D: Contract Value per lot (INR) ─────────────────────────────────────
#
# Contract Value (1 lot) = Settle Price (INR/unit) × units_per_lot
#
# Example:
#   Cotton Seed Oilcake: 361 × 500 = INR 1,80,500 per lot
#   Turmeric:          14100 × 50  = INR 7,05,000 per lot

df['contract_value_INR'] = df['settle_price'] * df['units_per_lot']

print('Contract value per lot (INR):')
display(df[['Commodity','settle_price','units_per_lot','contract_value_INR']]
        .style.format({'settle_price': '{:,.2f}',
                       'units_per_lot': '{:,.0f}',
                       'contract_value_INR': '₹{:,.0f}'}))

In [ ]:
# ── STEP 1E: Signed Lots (Long = +, Short = −) ────────────────────────────────
#
# SPAN needs to know direction to calculate P&L.
# Long position GAINS when price goes UP.
# Short position GAINS when price goes DOWN.

df['signed_lots'] = df.apply(
    lambda r: r['lots'] if r['direction'].strip().lower() == 'long' else -r['lots'],
    axis=1
)

print('Signed lots (+ = Long, − = Short):')
display(df[['Commodity','direction','lots','signed_lots']])

In [ ]:
# ── STEP 1F: Final Portfolio Summary ──────────────────────────────────────────

df['total_notional_INR'] = df['contract_value_INR'] * df['lots']

print('=' * 85)
print(f'  {"COMMODITY":<25} {"DIR":>5} {"LOTS":>5} {"SETTLE":>10} {"CONTRACT VAL/LOT":>18} {"TOTAL NOTIONAL":>15}')
print('=' * 85)
for _, r in df.iterrows():
    print(f"  {r['Commodity']:<25} {r['direction']:>5} {int(r['lots']):>5} "
          f"{r['settle_price']:>10,.2f} "
          f"{r['contract_value_INR']:>18,.0f} "
          f"{r['total_notional_INR']:>15,.0f}")
print('─' * 85)
print(f"  {'PORTFOLIO TOTAL':<25} {'':>5} {int(df['lots'].sum()):>5} {'':>10} {'':>18} "
      f"{df['total_notional_INR'].sum():>15,.0f}")
print('=' * 85)

# Cross-check against original notional column
print('\n── Notional Cross-check vs Excel ──')
df['notional_match'] = df.apply(
    lambda r: '✓ Match' if abs(r['total_notional_INR'] - r['notional_INR']) < 1 else
              f'⚠ Excel={r["notional_INR"]:,.0f}  Calc={r["total_notional_INR"]:,.0f}',
    axis=1
)
display(df[['Commodity', 'notional_INR', 'total_notional_INR', 'notional_match']])

print('\n✓ Step 1 complete. Portfolio is clean and ready for Step 2 (SPAN Parameters).')